In [3]:

import scanpy as sc
# import squidpy as sq
import pandas as pd
# import numpy as np
# import json
# import time
import anndata as ad

In [15]:
adata = "/nfs/home/students/m.back/swarm/backend/uploads/job_1775807036776_d857c27f-aa86-455a-8f71-f5875450a996/kackhaufen1/adata_tg_scores.h5ad"
adata = sc.read_h5ad(adata)

file_path = "/nfs/home/students/m.back/swarm/backend/calc_multiome_scores/multiome_tf_regulation/notebooks_pipeline/exmpl_results_breast/peak_stats_summary_smooth_muscle_contraction.csv"
df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

desired_cols = [
    "Gene",
    "Cluster",
    "Annotation",
    "Peak",
    "Class",
    "Link Score",
    "Link Z",
    "Link P",
    "Acc. T-stat",
    "Acc. FDR",
    "Accessible Cells",
    "P(expr|acc), cluster",
    "P(expr|acc), bg",
    "P(expr & acc), cluster",
    "P(expr & acc), all",
    "Enrichment, cluster",
    "Enrichment, all",
    "Delta P(expr|acc)",
    "Promoter Peaks",
    "Distal Peaks",
    "Pass Type",
]

existing_cols = [col for col in desired_cols if col in df.columns]
remaining_cols = [col for col in df.columns if col not in existing_cols]
df = df[existing_cols + remaining_cols]
peak_stats_dict = df.to_dict(orient="list")
adata.uns["peak_stats"] = df
adata.uns["peak_stats_dict"] = peak_stats_dict
    

In [17]:
file_path = "/nfs/home/students/m.back/swarm/backend/calc_multiome_scores/multiome_tf_regulation/notebooks_pipeline/exmpl_results_breast/motif_stats_summary_smooth_muscle_contraction.csv"
df = pd.read_csv(file_path)

# clean column names and string values
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

desired_cols = [
    "Gene",
    "Cluster",
    "TF",
    "Motif",
    "Prox Motif count",
    "Prox Bg count",
    "Prox Log2FC",
    "Prox p-value adj",
    "Dist Motif count",
    "Dist Bg count",
    "Dist Log2FC",
    "Dist p-value adj",
    "Prom Motif count",
    "FP Score",
    "Bg FP Score",
    "FP p-value adj",
    "Bg Size",
    "Flank sd",
    "Bg Flank sd",
    "Left Flank != 0",
    "Right Flank != 0",
]

existing_cols = [col for col in desired_cols if col in df.columns]
remaining_cols = [col for col in df.columns if col not in existing_cols]
df = df[existing_cols + remaining_cols]

for col in [
    "Cluster",
    "Prox Motif count",
    "Prox Bg count",
    "Prox Log2FC",
    "Prox p-value adj",
    "Dist Motif count",
    "Dist Bg count",
    "Dist Log2FC",
    "Dist p-value adj",
    "Prom Motif count",
    "FP Score",
    "Bg FP Score",
    "FP p-value adj",
    "Bg Size",
    "Flank sd",
    "Bg Flank sd",
    "Left Flank != 0",
    "Right Flank != 0",
]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

motif_stats_dict = df.to_dict(orient="list")

adata.uns["motif_stats"] = df
adata.uns["motif_stats_dict"] = motif_stats_dict


In [20]:
import json
graph_path = "/nfs/home/students/m.back/swarm/backend/calc_multiome_scores/multiome_tf_regulation/notebooks_pipeline/exmpl_results_breast/graph.json"

with open(graph_path, "r") as f:
    graph = json.load(f)

nodes_df = pd.DataFrame(graph["nodes"])
links_df = pd.DataFrame(graph["links"])

nodes_df.head()
links_df.head()

,source,target,edge_color,edge_width,edge_dash
0,77,33,red,1,0
1,78,33,red,1,0
2,61,33,red,1,"4,4"
3,21,33,yellow,1,"4,4"
4,44,33,yellow,1,0


In [29]:
adata.uns["nodes_df_motif_net"] = nodes_df
adata.uns["links_df_motif_net"] = links_df

In [34]:
adata.uns["nodes_df_motif_net"] = None
adata.uns["links_df_motif_net"] = None

In [1]:
import anndata as ad

In [4]:
ad.__version__

/tmp/ipykernel_1948885/1220773638.py:1: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  ad.__version__


'0.12.6'

In [35]:
import anndata as ad

ad.settings.allow_write_nullable_strings = True
adata.write_h5ad("/nfs/data3/mopitas/mapra/test_obj/adata_tg_scores_with_motif_gene_stuff.h5ad")

In [38]:
adata.uns.keys()

dict_keys(['chromvar_gearyC', 'chromvar_moranI', 'chromvar_motifs', 'diff_motif_activity_top_motifs', 'dissociated_masked_prob_column_map', 'dissociated_prob_column_map', 'hvg', 'neighbors', 'overlap_genes', 'rank_genes_groups', 'spatial', 'training_genes', 'peak_stats', 'peak_stats_dict', 'motif_stats', 'motif_stats_dict', 'nodes_df_motif_net', 'links_df_motif_net'])

In [5]:
adata_for_annika = ad.read_h5ad("/nfs/data3/mopitas/mapra/test_obj/adata_tg_scores_with_motif_gene_stuff.h5ad")

In [12]:
import anndata as ad

adata_to_save = adata_for_annika.copy()

# remove None entries from .uns if present
for key in ["nodes_df_motif_net", "links_df_motif_net"]:
    if key in adata_to_save.uns and adata_to_save.uns[key] is None:
        del adata_to_save.uns[key]

ad.settings.allow_write_nullable_strings = True
adata_to_save.write_h5ad(
    "/nfs/data3/mopitas/mapra/test_obj/adata_tg_scores_with_motif_gene_stuff_compat_0116.h5ad",
    compression="gzip",
)

In [14]:
adata_to_save.write_h5ad(
    "/nfs/home/students/m.back/swarm/backend/data/adata_tg_scores_with_motif_gene_stuff_compat_0116.h5ad",
    compression="gzip",
)

In [4]:
read_adata = ad.read_h5ad("/nfs/home/students/m.back/swarm/backend/data/adata_tg_scores_with_motif_gene_stuff_compat_0116.h5ad")

In [5]:
read_adata.uns.keys()

dict_keys(['chromvar_gearyC', 'chromvar_moranI', 'chromvar_motifs', 'diff_motif_activity_top_motifs', 'dissociated_masked_prob_column_map', 'dissociated_prob_column_map', 'hvg', 'motif_stats', 'motif_stats_dict', 'neighbors', 'overlap_genes', 'peak_stats', 'peak_stats_dict', 'rank_genes_groups', 'spatial', 'training_genes'])

In [7]:
peak_stats = read_adata.uns["peak_stats"]
motif_stats = read_adata.uns["motif_stats"]

In [10]:
read_adata.uns["peak_stats"] = {"0": peak_stats}
read_adata.uns["motif_stats"] = {"0": motif_stats}


In [16]:
ad.settings.allow_write_nullable_strings = True
read_adata.write_h5ad(
    "/nfs/data3/mopitas/mapra/test_obj/adata_tg_scores_with_motif_gene_stuff_compat_0114.h5ad",
    compression="gzip",
)

In [ ]:
read_adata = ad.read_h5ad("/workspaces/swarm/backend/data/adata_tg_scores_with_motif_gene_stuff_compat_0114.h5ad")

In [19]:
read_adata.uns["peak_stats"]["0"]

,Gene,Cluster,Annotation,Peak,Class,Link Score,Link Z,Link P,Acc. T-stat,Acc. FDR,...,"P(expr|acc), cluster","P(expr|acc), bg","P(expr & acc), cluster","P(expr & acc), all","Enrichment, cluster","Enrichment, all",Delta P(expr|acc),Promoter Peaks,Distal Peaks,Pass Type
0,MYLK,0,module_genes,chr3-123766211-123767342,distal,0.093637,3.276845,5.248694e-04,6.141285,2.363083e-09,...,0.686131,0.361446,0.061883,0.029502,1.018801,1.775816,0.324686,3,9,cluster_specific
1,MYLK,0,module_genes,chr3-123815972-123817188,distal,0.064807,3.966681,3.644017e-05,8.566264,7.019418e-17,...,0.716981,0.338462,0.075049,0.030268,1.064608,1.910201,0.378520,3,9,cluster_specific
2,MYLK,0,module_genes,chr3-123883475-123884734,proximal,0.050383,3.653168,1.295125e-04,8.727091,2.051297e-17,...,0.680851,0.251185,0.189598,0.085632,1.010961,1.478984,0.429666,3,9,cluster_specific
3,KCNMA1,0,module_genes,chr10-77636810-77638795,proximal,0.054927,4.224431,1.197725e-05,8.838366,9.343907e-18,...,0.636856,0.184971,0.154707,0.063410,1.073679,1.528471,0.451885,2,6,cluster_specific
4,MYH11,0,module_genes,chr16-15856768-15857482,proximal,0.065553,4.334615,7.300771e-06,5.760547,1.868771e-08,...,0.687500,0.243902,0.050691,0.020498,1.112154,1.787042,0.443598,6,6,cluster_specific
5,MYH11,0,module_genes,chr16-15857870-15858601,proximal,0.054931,3.406572,3.289211e-04,7.703362,6.008124e-14,...,0.711656,0.265432,0.076366,0.030460,1.151231,1.920139,0.446224,6,6,cluster_specific
6,KCNMA1,0,seed_genes,chr10-77636810-77638795,proximal,0.074482,2.930289,1.693235e-03,8.838366,9.343907e-18,...,0.636856,0.184971,0.154707,0.063410,1.073679,1.528471,0.451885,2,6,cluster_specific
7,MYH11,0,seed_genes,chr16-15856768-15857482,proximal,0.094247,1.802430,3.573887e-02,5.760547,1.868771e-08,...,0.687500,0.243902,0.050691,0.020498,1.112154,1.787042,0.443598,6,6,cluster_specific
8,MYH11,0,seed_genes,chr16-15857870-15858601,proximal,0.078460,1.726382,4.213940e-02,7.703362,6.008124e-14,...,0.711656,0.265432,0.076366,0.030460,1.151231,1.920139,0.446224,6,6,cluster_specific
9,MYH11,0,seed_genes,chr16-15856768-15857482,proximal,0.104215,3.785588,7.667278e-05,5.760547,1.868771e-08,...,0.687500,0.243902,0.050691,0.020498,1.112154,1.787042,0.443598,6,6,cluster_specific
